# Topolojik Uzaylar ve Yüklemler

**Modüller:** `pytop.spaces`, `pytop.predicates`, `pytop.sets`, `pytop.topology_builders`  
**Konu:** Topolojik uzay tanımı, açık/kapalı kümeler, yoğunluk, hiçbir yerde yoğun olmama

---

Bu notebook altı bölümden oluşur:
1. **Konu** — Topoloji nedir, neden açık kümeler?
2. **Teoremler** — Kapalı kümeler, yoğunluk, taban (tam ispat)
3. **Fonksiyon Analizi** — Pseudo-kod, karmaşıklık, avantaj/dezavantaj
4. **API** — `TopologicalSpace`, `is_open_subset`, `analyze_predicate`, topoloji kurucuları
5. **Örnekler** — Ayrık topolojiden Sierpiński uzayına dört senaryo
6. **Alıştırmalar** — Kodlama, teori ve karşılaştırma

---
## 1. Konu

### 1.1 Sezgisel Açıklama

Analiz dersinde süreklilik şöyle öğretilir: $f: \mathbb{R} \to \mathbb{R}$ her $\varepsilon > 0$ için bir $\delta > 0$ bulunabiliyorsa süreklidir. Bu tanım mesafeye dayanır. Peki mesafe olmayan bir uzayda süreklilik ne anlama gelir?

Topoloji bu soruya şöyle yanıt verir: süreklilik için tek ihtiyacımız, uzayın "hangi kümeler açıktır" bilgisidir. Mesafe silinir, yerine bir **açık kümeler ailesi** konur.

**Sezgi:**  
- "Açık küme" = yakın noktaları birlikte tutan bir bölge  
- Uzayın topolojisi = hangi bölgelerin "açık" sayıldığı  
- Aynı küme üzerinde farklı topolojiler → farklı süreklilik kavramları

**Üç temel örnek:**
| Topoloji | Açık kümeler | Özellik |
|----------|-------------|----------|
| İnce (ayrık) | Tüm alt kümeler | Her fonksiyon sürekli |
| Kaba (indirgenmiş) | $\{\emptyset, X\}$ | Sadece sabit fonksiyonlar sürekli |
| Öklid | Açık aralıkların birleşimleri | Klasik analiz |

---

### 1.2 Formal Tanımlar

**Topolojik Uzay:**  
Bir $X$ kümesi ve $\tau \subseteq \mathcal{P}(X)$ ailesi; $(X, \tau)$ bir topolojik uzaydır ancak ve ancak:
1. $\emptyset \in \tau$ ve $X \in \tau$
2. $U, V \in \tau \Rightarrow U \cap V \in \tau$ \hspace{1em}(sonlu kesişim)
3. $\{U_\alpha\}_{\alpha \in I} \subseteq \tau \Rightarrow \bigcup_\alpha U_\alpha \in \tau$ \hspace{1em}(keyfi birleşim)

$\tau$ elemanlarına **açık küme** denir. $\tau$'ya uzayın **topolojisi** denir.

**Kapalı Küme:**  
$F \subseteq X$ kapalıdır $\iff$ $X \setminus F$ açıktır.

**Yoğun Alt Küme:**  
$A \subseteq X$ yoğundur $\iff$ $\overline{A} = X$, yani her boş olmayan açık $U$, $A \cap U \ne \emptyset$.

**Hiçbir Yerde Yoğun Olmayan:**  
$A$ hiçbir yerde yoğun değildir $\iff$ $\overline{A}$'nın içi boştur, yani $\operatorname{int}(\overline{A}) = \emptyset$.

**Clopen:**  
$A$ hem açık hem kapalıdır. Ayrık topolojide her küme clopen; indirgenmiş topolojide yalnızca $\emptyset$ ve $X$ clopen.

---
## 2. Teoremler

### Teorem 1 — Kapalı Kümeler Ailesi

**İfade:**  
$(X, \tau)$ topolojik uzay, $\mathcal{F} = \{F \subseteq X : X \setminus F \in \tau\}$ kapalı kümeler ailesi olsun. O zaman:
1. $\emptyset, X \in \mathcal{F}$
2. $F_1, F_2 \in \mathcal{F} \Rightarrow F_1 \cup F_2 \in \mathcal{F}$
3. $\{F_\alpha\} \subseteq \mathcal{F} \Rightarrow \bigcap_\alpha F_\alpha \in \mathcal{F}$

**İspat:**  
1. $X \setminus \emptyset = X \in \tau$ ve $X \setminus X = \emptyset \in \tau$, dolayısıyla $\emptyset, X \in \mathcal{F}$.
2. $F_1, F_2 \in \mathcal{F}$ ise $X \setminus F_i \in \tau$. De Morgan: $X \setminus (F_1 \cup F_2) = (X \setminus F_1) \cap (X \setminus F_2)$. Sonlu kesişim aksiyomu (T2) gereği bu küme açıktır; dolayısıyla $F_1 \cup F_2 \in \mathcal{F}$.
3. Her $\alpha$ için $X \setminus F_\alpha \in \tau$. De Morgan: $X \setminus \bigcap_\alpha F_\alpha = \bigcup_\alpha (X \setminus F_\alpha)$. Keyfi birleşim aksiyomu (T3) gereği açıktır; dolayısıyla $\bigcap_\alpha F_\alpha \in \mathcal{F}$. $\square$

---

### Teorem 2 — Yoğunluğun Açık Küme Karakterizasyonu

**İfade:**  
$(X, \tau)$ topolojik uzay, $A \subseteq X$. Şu ifadeler denktir:
1. $A$ yoğundur ($\overline{A} = X$)
2. Her boş olmayan $U \in \tau$ için $A \cap U \ne \emptyset$
3. $A$'yı içeren en küçük kapalı küme $X$'in kendisidir

**İspat ($1 \Leftrightarrow 2$):**  
Kapanış tanımı: $\overline{A} = \{x \in X : \forall U \in \tau,\, x \in U \Rightarrow A \cap U \ne \emptyset\}$.  
$\overline{A} = X$ ise her $x \in X$ ve her açık $U \ni x$ için $A \cap U \ne \emptyset$. Özellikle her boş olmayan açık küme bir nokta içerdiğinden (2) sağlanır.  
Tersine, (2) varsa her $x$ ve $x$'i içeren her $U$ için $A \cap U \ne \emptyset$, dolayısıyla $x \in \overline{A}$, yani $\overline{A} = X$. $\square$

---

### Teorem 3 — Taban Karakterizasyonu

**İfade:**  
$\mathcal{B} \subseteq \tau$ bir **taban**dır (her $U \in \tau$, $\mathcal{B}$ elemanlarının birleşimidir) ancak ve ancak:
1. $\bigcup_{B \in \mathcal{B}} B = X$
2. Her $B_1, B_2 \in \mathcal{B}$ ve $x \in B_1 \cap B_2$ için $\exists B_3 \in \mathcal{B}$ ile $x \in B_3 \subseteq B_1 \cap B_2$

**İspat (⇒):**  
Eğer $\mathcal{B}$ tabansa: $X$ açık olduğundan $X = \bigcup B_\alpha$ — (1) sağlanır. $B_1 \cap B_2$ açıktır (T2), dolayısıyla $\mathcal{B}$ elemanlarının birleşimidir; her $x \in B_1 \cap B_2$ için uygun $B_3$ vardır — (2) sağlanır.

**İspat (⇐):**  
(1) ve (2) varsayılsın. $\tau' = \{\bigcup \mathcal{B}' : \mathcal{B}' \subseteq \mathcal{B}\}$ tanımla. (2) gereği $\tau'$ sonlu kesişime kapalıdır (standart argüman). Dolayısıyla $\tau'$ bir topoloji ve $\mathcal{B}$ onun tabanıdır. $\square$

---
## 4. API

In [ ]:
from pytop.predicates import is_open_subset, is_closed_subset, is_dense_subset, analyze_predicate
from pytop.sets import make_set
from pytop.topology_builders import discrete_topology, make_topology

# is_open_subset: O(1) hash lookup
disc = discrete_topology(1, 2, 3, 4, 5)
A = make_set(2, 3)
print(f"Ayrık topolojide {{2,3}} açık: {is_open_subset(disc, A).is_true}")   # True

# is_dense_subset: O(|τ| * |A|)
# 5 noktalı uzay: τ = {∅, {1}, {2,3}, {1,2,3}, X}
sp5 = make_topology({1,2,3,4,5}, {1}, {2,3}, {1,2,3})
# {2,4} yoğun mu? Her boş olmayan açık kümeyle kesişiyor mu?
B = make_set(2, 4)
r_dense = is_dense_subset(sp5, B)
print(f"{{2,4}} yoğun: {r_dense.is_true}")

# analyze_predicate: dispatcher
print("\nanalyze_predicate çıktıları:")
for pred in ["open", "closed", "dense", "nowhere_dense"]:
    r = analyze_predicate(sp5, make_set(1), pred)
    icon = "✓" if r.is_true else "✗"
    print(f"  {pred:15} {icon}  {r.justification[0][:60]}")

---
## 3. API

---
## 5. Örnekler

### Örnek 1 — Minimal: Ayrık Topoloji

### Küme ve Topoloji Kurucuları

| Fonksiyon | Sözdizimi | Döndürür |
|-----------|-----------|----------|
| `make_set(*elemanlar)` | `make_set(1, 2, 3)` | `frozenset` |
| `discrete_topology(*elemanlar)` | `discrete_topology(1, 2, 3)` | `FiniteTopologicalSpace` |
| `indiscrete_topology(*elemanlar)` | `indiscrete_topology(1, 2, 3)` | `FiniteTopologicalSpace` |
| `sierpinski_space()` | `sierpinski_space()` | `FiniteTopologicalSpace` |
| `make_topology(carrier, *açık_kümeler)` | `make_topology({1,2,3}, {1}, {2,3})` | `FiniteTopologicalSpace` |

`make_topology`'de ∅ ve X otomatik eklenir; yalnızca ekstra açık kümeleri belirtmek yeter.

### Yüklem Fonksiyonları

| Fonksiyon | Alınan argümanlar | Döndürür |
|-----------|------------------|----------|
| `is_open_subset(sp, A)` | uzay, alt küme | `Result` |
| `is_closed_subset(sp, A)` | uzay, alt küme | `Result` |
| `is_clopen_subset(sp, A)` | uzay, alt küme | `Result` |
| `is_dense_subset(sp, A)` | uzay, alt küme | `Result` |
| `analyze_predicate(sp, A, name)` | uzay, alt küme, `"open"`/`"closed"`/`"dense"`/`"clopen"`/`"nowhere_dense"` | `Result` |

---
## 4. Örnekler

### Örnek 1 — Minimal: Ayrık Topoloji

In [ ]:
discrete = discrete_topology(1, 2, 3)

A = make_set(2)
r_open   = is_open_subset(discrete, A)
r_closed = is_closed_subset(discrete, A)
r_clopen = is_clopen_subset(discrete, A)

print(f"Ayrık topoloji — {{2}} açık mı?   {r_open.is_true}")
print(f"Ayrık topoloji — {{2}} kapalı mı? {r_closed.is_true}")
print(f"Ayrık topoloji — {{2}} clopen mı? {r_clopen.is_true}")

### Örnek 2 — Orta Düzey: Sierpiński Uzayı

In [ ]:
sierpinski = sierpinski_space()

for subset in [make_set(0), make_set(1), make_set(0, 1), make_set()]:
    o = is_open_subset(sierpinski, subset)
    c = is_closed_subset(sierpinski, subset)
    print(f"  {str(set(subset)):8} → açık={o.is_true}, kapalı={c.is_true}")

### Örnek 3 — Gelişmiş: Yoğunluk Analizi

In [ ]:
# 4 noktalı uzay: {a,b,c,d}, τ = {∅, {a}, {b,c}, {a,b,c}, X}
sp4 = make_topology({'a', 'b', 'c', 'd'}, {'a'}, {'b', 'c'}, {'a', 'b', 'c'})

# {a,d} yoğun mu? Her açık kümeyle kesişiyor mu?
A_dense = make_set('a', 'd')
r = is_dense_subset(sp4, A_dense)
print(f"{{a,d}} yoğun mu? {r.is_true}")
print(f"Gerekçe: {r.justification[0]}")

# {d} tek başına yoğun mu?
r2 = is_dense_subset(sp4, make_set('d'))
print(f"\n{{d}} yoğun mu? {r2.is_true}")
if r2.justification:
    print(f"Gerekçe: {r2.justification[0]}")

---
## 6. Alıştırmalar

### Alıştırma 1 — Kodlama Görevi

$X = \{1, 2, 3, 4\}$ üzerinde **indirgenmiş (indiscrete) topoloji**yi oluşturun: $\tau = \{\emptyset, X\}$.

Sonra şunu doğrulayın:
- $\{1\}$ açık değildir
- $\{1\}$ kapalı değildir  
- $\{1, 2\}$ yoğundur (her boş olmayan açık kümeyle kesiştiğinden)

In [ ]:
# Bir alt kümenin tüm topolojik özelliklerini tek fonksiyonla sorgula
for predicate in ["open", "closed", "clopen", "dense", "nowhere_dense"]:
    result = analyze_predicate(sp4, make_set('b', 'c'), predicate)
    icon = "✓" if result.is_true else ("✗" if result.is_false else "?")
    print(f"  {predicate:15} {icon}  {result.justification[0][:60]}")

---
## 5. Alıştırmalar

### Alıştırma 1 — Kodlama Görevi

$X = \{1, 2, 3, 4\}$ üzerinde **indirgenmiş (indiscrete) topoloji**yi oluşturun: $\tau = \{\emptyset, X\}$.

Sonra şunu doğrulayın:
- $\{1\}$ açık değildir
- $\{1\}$ kapalı değildir  
- $\{1, 2\}$ yoğundur (her boş olmayan açık kümeyle kesiştiğinden)

In [ ]:
indiscrete = indiscrete_topology(1, 2, 3, 4)

print(is_open_subset(indiscrete, make_set(1)).is_true)        # False bekleniyor
print(is_closed_subset(indiscrete, make_set(1)).is_true)      # False bekleniyor
print(is_dense_subset(indiscrete, make_set(1, 2)).is_true)    # True bekleniyor

### Alıştırma 2 — Teori Sorusu

a) Topolojik uzayın 3 aksiyomunu yazın. Her aksiyom olmadan topoloji ne kaybeder?  
b) İndirgenmiş (indiscrete) topolojide hangi kümeler yoğundur? Neden?  
c) Ayrık topolojide hiçbir yerde yoğun olmayan kümeler hangileridir?

_Yanıtlarınızı buraya yazın:_

a) ...

b) ...

c) ...

### Alıştırma 4 — Topoloji Boyutu ve Sorgu Maliyeti

a) $X = \{1,2,3,4\}$ üzerinde ayrık topoloji oluşturun. `|τ|` kaçtır? `is_open_subset` bu büyük `τ`'da neden yine de O(1)?

b) İndirgenmiş topolojide (`τ = {∅, X}`) yoğunluk testi ne kadar sürer? `is_dense_subset` için O(\|τ\|) bağıntısını `τ` büyüklüğüne göre yorumlayın.

c) `analyze_predicate` fonksiyonunu beş farklı küme üzerinde (`∅`, `{1}`, `{2,3}`, `{4}`, `X`) çalıştırın. Her birinin hangi predicate değerlerine sahip olduğunu bir tabloda gösterin.

In [ ]:
from pytop.predicates import analyze_predicate
from pytop.sets import make_set
from pytop.topology_builders import discrete_topology, indiscrete_topology, make_topology

# a) Ayrık topoloji boyutu
disc4 = discrete_topology(1, 2, 3, 4)
print(f"|τ| ayrık (n=4): {len(disc4.topology)}  (2^4 = {2**4})")
print(f"is_open O(1): τ frozenset[frozenset] olduğu için hash lookup")

# b) İndirgenmiş topolojide yoğunluk
indisc4 = indiscrete_topology(1, 2, 3, 4)
print(f"\n|τ| indirgenmiş: {len(indisc4.topology)}  (sadece ∅ ve X)")
from pytop.predicates import is_dense_subset
r = is_dense_subset(indisc4, make_set(1))
print(f"{{1}} yoğun (indirgenmiş): {r.is_true}  (boş olmayan tek açık küme X)")

# c) Beş küme için predicate tablosu
sp_ex = make_topology({1,2,3,4}, {1}, {2,3}, {1,2,3})
subsets = [
    ("∅",       make_set()),
    ("{1}",     make_set(1)),
    ("{2,3}",   make_set(2,3)),
    ("{4}",     make_set(4)),
    ("X",       make_set(1,2,3,4)),
]
preds = ["open", "closed", "dense", "nowhere_dense"]
print(f"\n{'Küme':10}" + "".join(f"{p[:9]:11}" for p in preds))
print("-" * (10 + 11*len(preds)))
for label, A in subsets:
    row = f"{label:10}"
    for p in preds:
        r = analyze_predicate(sp_ex, A, p)
        row += f"{'✓' if r.is_true else '✗':11}"
    print(row)

---
## Özet

| Kavram | Ana Sonuç |
|--------|-----------|
| Topolojik uzay | $(X, \tau)$; aksiyomlar: $\emptyset, X \in \tau$; sonlu kesişim; keyfi birleşim |
| `is_open_subset` | O(1) — `τ ⊆ frozenset[frozenset]`; hash lookup |
| `is_closed_subset` | O(n) tümleyen + O(1) lookup |
| `is_dense_subset` | O(\|τ\| · \|A\|) — her açık küme ile kesişim kontrolü |
| `analyze_predicate` | Dispatcher; `name` → uygun `is_*` fonksiyonu |
| Kapalı kümeler | De Morgan ile $\tau$'nun aksiyomlarını sağlar |
| Yoğunluk | $\overline{A} = X$ ↔ her boş olmayan açık kümeyle kesişir |

**Bir sonraki adım:** `pytop.compactness` — kompaktlık, Lindelöf, sıralı kompaktlık.

### Alıştırma 3 — Karşılaştırma

Ayrık topoloji ile indirgenmiş topolojiyi karşılaştırın:
1. `analyze_predicate` kullanarak her iki topolojide de $\{1\}$ kümesini analiz edin.
2. Sonuçlar neden farklı? Hangi aksiyom bu farkı yaratıyor?

In [ ]:
discrete   = discrete_topology(1, 2, 3)
indiscrete = indiscrete_topology(1, 2, 3)

A = make_set(1)
for pred in ["open", "closed", "dense"]:
    d_r = analyze_predicate(discrete,   A, pred)
    i_r = analyze_predicate(indiscrete, A, pred)
    icon_d = "✓" if d_r.is_true else "✗"
    icon_i = "✓" if i_r.is_true else "✗"
    print(f"  {pred:8} | ayrık={icon_d}  indirgenmiş={icon_i}")